In [1]:
!pip install -q sentence-transformers

In [2]:
pip install -U bitsandbytes -q

Note: you may need to restart the kernel to use updated packages.


In [ ]:
!pip install -q gensim

In [4]:
import os
import json
import random
import numpy as np
from collections import Counter
from typing import Optional
import torch
from datasets import load_dataset
import random
import time
import math

from sentence_transformers import SentenceTransformer

from dataclasses import dataclass, field

c:\Users\yash.paudel\AppData\Local\anaconda3\envs\sashuv-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [6]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
assert DEVICE == "cuda", "Switch to GPU runtime: Runtime > Change runtime type > T4 GPU"

Device: cuda


In [7]:
from model import load_sentence_encoder, load_model, generate_response, generate_batch
from scoring import clean_response, compute_stability, is_uncertainty_response, plateau_stop, responses_are_substantive
from data import load_triviaqa, load_mmlu, load_webquestions
from evaluate import evaluate, run_full_evaluation, print_comparison_table
from calibration import compute_qhat, check_coverage, run_calibration
from sampler import adaptive_sample, check_coverage, build_prediction_set
from helpers import _assign_split, _print_split_counts, filter_split, clear_embed_cache, embed_cached
from scoring import clean_response, compute_stability, plateau_stop, is_uncertainty_response, responses_are_substantive, token_f1
from config import QASample, SamplerConfig, UNCERTAINTY_TOKENS, COVERAGE_SIM_THRESHOLD, MMLU_SUBJECTS

In [8]:
model, tokenizer = load_model()

Loading mistralai/Mistral-7B-Instruct-v0.3 ...


`torch_dtype` is deprecated! Use `dtype` instead!
W0406 11:09:17.809000 11240 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights:   1%|          | 2/291 [00:00<00:43,  6.67it/s]c:\Users\yash.paudel\AppData\Local\anaconda3\envs\sashuv-env\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 291/291 [00:06<00:00, 42.99it/s]


Model loaded.
GPU memory used: 4.1 GB


In [9]:
triviaqa_samples   = load_triviaqa(SEED, n_total=3000)
webq_samples       = load_webquestions(SEED)
mmlu_samples       = load_mmlu(SEED)

DATASETS = {
    "triviaqa": triviaqa_samples,
    "webq":     webq_samples,
    "mmlu":     mmlu_samples,
}

# Quick sanity check
for name, samples in DATASETS.items():
    cal  = len(filter_split(samples, "calibration"))
    val  = len(filter_split(samples, "validation"))
    test = len(filter_split(samples, "test"))
    print(f"{name}: cal={cal}, val={val}, test={test}")

TriviaQA loaded: 3000 samples
  calibration: 1500
  validation: 750
  test: 750
WebQuestions loaded: 3778 samples
  calibration: 1889
  validation: 945
  test: 944
MMLU loaded: 2314 samples across 5 subjects
  calibration: 1157
  validation: 579
  test: 578
triviaqa: cal=1500, val=750, test=750
webq: cal=1889, val=945, test=944
mmlu: cal=1157, val=579, test=578


In [ ]:
# def test_semantic_coverage():
#     print("Testing semantic check_coverage...\n")

#     assert check_coverage([], ["shakespeare"]) == True
#     print("  PASS  abstention → covered")

#     assert check_coverage(["eastern"], ["north american eastern time zone"]) == True
#     print("  PASS  'eastern' covers 'north american eastern time zone'")

#     assert check_coverage(["durbin"], ["dick durbin"]) == True
#     print("  PASS  'durbin' covers 'dick durbin'")

#     assert check_coverage(["shakespeare"], ["william shakespeare"]) == True
#     print("  PASS  'shakespeare' covers 'william shakespeare'")

#     assert check_coverage(["napoleon"], ["shakespeare"]) == False
#     print("  PASS  'napoleon' does not cover 'shakespeare'")

#     assert check_coverage(["kansas"], ["united states of america"]) == False
#     print("  PASS  'kansas' does not cover 'united states of america' (genuinely wrong)")

#     print("\nAll semantic coverage tests passed.\n")

# test_semantic_coverage()

In [ ]:
# def is_uncertainty_response(response: str) -> bool:
#     """Return True if response expresses ignorance rather than a fact."""
#     r = response.lower().strip()
#     return any(token in r for token in UNCERTAINTY_TOKENS)

# def responses_are_substantive(cleaned_responses: list,
#                                uncertainty_threshold: float = 0.6) -> bool:
#     """
#     Return True if the majority of responses are substantive answers.

#     If more than uncertainty_threshold fraction are uncertainty expressions,
#     treat the response set as non-substantive → force abstention.
#     """
#     if not cleaned_responses:
#         return False

#     n_uncertain = sum(1 for r in cleaned_responses
#                       if is_uncertainty_response(r))
#     return (n_uncertain / len(cleaned_responses)) < uncertainty_threshold

In [10]:
DEFAULT_CONFIG = SamplerConfig()

In [ ]:
# def run_integration_test():
#     cfg = DEFAULT_CONFIG

#     test_cases = [
#         (
#             "Who wrote the play Hamlet?",
#             ["shakespeare", "william shakespeare"],
#             "EASY",
#             "expect: abstain=False, reason=plateau, N~0.04, covered=True",
#         ),
#         (
#             "What caused the fall of the Roman Empire?",
#             ["decline", "barbarian invasions", "multiple factors"],
#             "MEDIUM",
#             "expect: abstain=True, reason=budget_exhausted",
#         ),
#         (
#             "What was the exact population of the Byzantine Empire in 500 AD?",
#             ["unknown"],
#             "HARD",
#             "expect: abstain=True, reason=uncertainty_responses",
#         ),
#     ]

#     print(f"\nINTEGRATION TEST\n{'='*65}")

#     for question, gold, label, expectation in test_cases:
#         clear_embed_cache()
#         result = adaptive_sample(question, config=cfg)
#         covered = check_coverage(
#             result["cleaned"] if not result["abstain"] else [],
#             gold,
#         )

#         print(f"\n[{label}] {expectation}")
#         print(f"  Q        : {question}")
#         print(f"  abstain  : {result['abstain']}")
#         print(f"  reason   : {result['reason']}")
#         print(f"  n_score  : {result['n_score']}")
#         print(f"  batches  : {result['n_batches']}/{cfg.max_batches}")
#         print(f"  stability: {result['stability']:.3f}")
#         print(f"  history  : {[round(h, 3) for h in result['history']]}")
#         print(f"  responses: {list(set(result['cleaned']))[:5]}")
#         print(f"  covered  : {covered}")

# run_integration_test()


In [ ]:
# def test_compute_qhat():
#     print("Testing compute_qhat...\n")

#     # Test 1: worked example from earlier explanation
#     # n=5, alpha=0.1 → position = ceil(6 * 0.9) = ceil(5.4) = 6 > 5 → inf
#     scores = [0.01, 0.03, 0.29, 0.42, 0.69]
#     q = compute_qhat(scores, alpha=0.1)
#     assert q == float("inf"), f"Expected inf for n=5, got {q}"
#     print(f"  PASS  n=5, alpha=0.1 → q_hat=inf (position > n)")

#     # Test 2: n=9, alpha=0.1 → position = ceil(10 * 0.9) = ceil(9.0) = 9
#     # sorted[8] = max score
#     scores = [0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70]
#     q = compute_qhat(scores, alpha=0.1)
#     assert q == 0.70, f"Expected 0.70, got {q}"
#     print(f"  PASS  n=9, alpha=0.1 → q_hat={q:.2f} (9th of 9)")

#     # Test 3: n=100, alpha=0.1 → position = ceil(101 * 0.9) = ceil(90.9) = 91
#     import random
#     random.seed(42)
#     scores = sorted([random.random() for _ in range(100)])
#     q = compute_qhat(scores, alpha=0.1)
#     assert q == scores[90], f"Expected scores[90], got {q}"
#     print(f"  PASS  n=100, alpha=0.1 → q_hat=scores[90]={q:.4f}")

#     # Test 4: n=500, alpha=0.1 → position = ceil(501 * 0.9) = ceil(450.9) = 451
#     scores_500 = sorted([random.random() for _ in range(500)])
#     q = compute_qhat(scores_500, alpha=0.1)
#     assert q == scores_500[450], f"Expected scores_500[450]"
#     print(f"  PASS  n=500, alpha=0.1 → q_hat=scores[450]={q:.4f}")

#     # Test 5: alpha=0.2 → position = ceil(501 * 0.8) = ceil(400.8) = 401
#     q_lenient = compute_qhat(scores_500, alpha=0.2)
#     assert q_lenient <= q, "More lenient alpha should give smaller or equal q_hat"
#     print(f"  PASS  alpha=0.2 gives smaller q_hat ({q_lenient:.4f} <= {q:.4f})")

#     print("\nAll compute_qhat tests passed.\n")

# test_compute_qhat()

# tesstttt


In [11]:
LOFREEFCP_TRIVIAQA = {
    0.20: {"ECR": 80.1, "SSC": 79.0, "APSS": 2.19},
    0.25: {"ECR": 75.3, "SSC": 74.5, "APSS": 1.43},
    0.30: {"ECR": 70.3, "SSC": 76.7, "APSS": 1.08},
    0.35: {"ECR": 65.1, "SSC": 78.5, "APSS": 0.90},
    0.40: {"ECR": 60.0, "SSC": 81.0, "APSS": 0.75},
    0.45: {"ECR": 55.2, "SSC": 84.1, "APSS": 0.66},
}
LOFREEFCP_WEBQ = {
    0.35: {"ECR": 65.1, "SSC": 61.1, "APSS": 5.33},
    0.40: {"ECR": 60.0, "SSC": 60.0, "APSS": 2.68},
    0.45: {"ECR": 55.1, "SSC": 60.1, "APSS": 1.60},
    0.50: {"ECR": 50.3, "SSC": 59.9, "APSS": 1.06},
}

In [ ]:
TRIVIAQA_ALPHAS = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45]

triviaqa_results = run_full_evaluation(
    model=model,
    tokenizer=tokenizer,
    DATASETS=DATASETS,
    dataset_name="triviaqa",
    alphas=TRIVIAQA_ALPHAS,              # or whichever value you need
    lofree_ref=LOFREEFCP_TRIVIAQA,               # example
    cal_samples  = 500,
    test_samples = 500,

    
)

print_comparison_table(triviaqa_results, LOFREEFCP_TRIVIAQA, "TriviaQA")

# WebQ — match LofreeCP's alpha range
WEBQ_ALPHAS = [0.35, 0.40, 0.45, 0.50]

webq_results = run_full_evaluation(
    model = model,
    tokenizer = tokenizer,
    DATASETS = DATASETS,
    dataset_name="webq",
    alphas = WEBQ_ALPHAS,
    lofree_ref = LOFREEFCP_WEBQ,
    cal_samples  = 500,
    test_samples = 500,
)

print_comparison_table(webq_results, LOFREEFCP_WEBQ, "WebQuestions")

# Save all results
import json
with open("/content/final_results.json", "w") as f:
    # Strip full result lists for storage
    save = []
    for r in triviaqa_results + webq_results:
        d = {k: v for k, v in r.items() if k != "results"}
        save.append(d)
    json.dump(save, f, indent=2)
print("\nSaved to /content/final_results.json")



FULL EVALUATION: TRIVIAQA

--- alpha=0.35 (target ECR >= 65%) ---

Calibrating on triviaqa (n=500, alpha=0.35)
Config: SamplerConfig(batch_size=3, max_batches=6, eps=0.03, temperature=0.9, min_batches=2, min_stability=0.7)
-------------------------------------------------------


c:\Users\yash.paudel\AppData\Local\anaconda3\envs\sashuv-env\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Loading sentence encoder...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7358.18it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoder loaded. Embedding dim: 384


KeyboardInterrupt: 